# Kafka Producer — Yelp User Dataset (Multi-pass CDC)

## Strategy
- **Pass 1**: Full load — bắn toàn bộ ~2M users → Bronze INSERT all
- **Pass 2**: Update 30% users ngẫu nhiên với `review_count` và `fans` tăng → Silver MERGE trigger UPDATE
- **Pass 3**: Update 10% users (subset của Pass 2) thêm lần nữa → tạo version 3 trong SCD2

Mỗi pass gửi vào cùng topic `raw_yelp_users`. Bronze stream tiêu thụ liên tục.
Silver stream dùng MERGE INTO để detect thay đổi và ghi lịch sử SCD Type 2.

> Notebook đã module hóa — toàn bộ logic core nằm trong package `src/`.
> Notebook chỉ setup SparkSession, gọi function, và làm phần interactive (monitor/validate).


In [1]:
from pathlib import Path
import sys

project_root = Path.cwd().parent
sys.path.append(str(project_root))
# Install dependencies
!pip install kafka-python --quiet


[notice] A new release of pip is available: 23.0.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


In [2]:
from src.kafka_utils import get_producer
from src.producer import load_users, run_pass1, run_pass2, run_pass3
from src import config
import time

producer = get_producer()
print("✅ Kafka Producer khởi tạo thành công")
print(f"   Topic  : {config.KAFKA_TOPIC_YELP_USERS}")
print(f"   Broker : {config.KAFKA_BOOTSTRAP_SERVERS}")

/tmp/ipykernel_444/1066643969.py:6: DeprecationWarning: value_serializer does not implement kafka.serializer.Serializer
  producer = get_producer()


✅ Kafka Producer khởi tạo thành công
   Topic  : raw_yelp_users
   Broker : kafka:9092


In [3]:
# ============================================================
# PASS 1 — Full load toàn bộ users
# Đây là initial snapshot: Bronze nhận toàn bộ, Silver INSERT all
# ============================================================
print("=" * 60)
print("Đang đọc file user.json... (có thể mất 30-60s)")
all_users = load_users()
print(f"✅ Đọc xong: {len(all_users):,} users")
print()

run_pass1(producer, all_users, delay=0.0)

print()
print("⏸  Chờ 30s để Bronze pipeline xử lý xong trước khi Pass 2...")
time.sleep(30)

Đang đọc file user.json... (có thể mất 30-60s)


✅ Đọc xong: 1,987,897 users

PASS 1 — FULL LOAD (1,987,897 users)
  [PASS1]      500 / 1,987,897  (1,521 rec/s)
  [PASS1]    1,000 / 1,987,897  (2,758 rec/s)
  [PASS1]    1,500 / 1,987,897  (3,822 rec/s)
  [PASS1]    2,000 / 1,987,897  (4,728 rec/s)
  [PASS1]    2,500 / 1,987,897  (5,506 rec/s)
  [PASS1]    3,000 / 1,987,897  (6,216 rec/s)
  [PASS1]    3,500 / 1,987,897  (6,866 rec/s)
  [PASS1]    4,000 / 1,987,897  (7,338 rec/s)
  [PASS1]    4,500 / 1,987,897  (7,770 rec/s)
  [PASS1]    5,000 / 1,987,897  (8,200 rec/s)
  [PASS1]    5,500 / 1,987,897  (8,608 rec/s)
  [PASS1]    6,000 / 1,987,897  (8,947 rec/s)
  [PASS1]    6,500 / 1,987,897  (9,131 rec/s)
  [PASS1]    7,000 / 1,987,897  (9,390 rec/s)
  [PASS1]    7,500 / 1,987,897  (9,608 rec/s)
  [PASS1]    8,000 / 1,987,897  (9,836 rec/s)
  [PASS1]    8,500 / 1,987,897  (10,072 rec/s)
  [PASS1]    9,000 / 1,987,897  (10,314 rec/s)
  [PASS1]    9,500 / 1,987,897  (10,514 rec/s)
  [PASS1]   10,000 / 1,987,897  (10,716 rec/s)
  [PASS1] 

  [PASS1]  702,000 / 1,987,897  (17,337 rec/s)
  [PASS1]  702,500 / 1,987,897  (17,334 rec/s)
  [PASS1]  703,000 / 1,987,897  (17,333 rec/s)
  [PASS1]  703,500 / 1,987,897  (17,333 rec/s)
  [PASS1]  704,000 / 1,987,897  (17,331 rec/s)
  [PASS1]  704,500 / 1,987,897  (17,329 rec/s)
  [PASS1]  705,000 / 1,987,897  (17,329 rec/s)
  [PASS1]  705,500 / 1,987,897  (17,328 rec/s)
  [PASS1]  706,000 / 1,987,897  (17,327 rec/s)
  [PASS1]  706,500 / 1,987,897  (17,326 rec/s)
  [PASS1]  707,000 / 1,987,897  (17,324 rec/s)
  [PASS1]  707,500 / 1,987,897  (17,323 rec/s)
  [PASS1]  708,000 / 1,987,897  (17,322 rec/s)
  [PASS1]  708,500 / 1,987,897  (17,321 rec/s)
  [PASS1]  709,000 / 1,987,897  (17,318 rec/s)
  [PASS1]  709,500 / 1,987,897  (17,314 rec/s)
  [PASS1]  710,000 / 1,987,897  (17,313 rec/s)
  [PASS1]  710,500 / 1,987,897  (17,311 rec/s)
  [PASS1]  711,000 / 1,987,897  (17,312 rec/s)
  [PASS1]  711,500 / 1,987,897  (17,310 rec/s)
  [PASS1]  712,000 / 1,987,897  (17,309 rec/s)
  [PASS1]  71

  [PASS1]  826,500 / 1,987,897  (17,044 rec/s)
  [PASS1]  827,000 / 1,987,897  (17,041 rec/s)
  [PASS1]  827,500 / 1,987,897  (17,040 rec/s)
  [PASS1]  828,000 / 1,987,897  (17,041 rec/s)
  [PASS1]  828,500 / 1,987,897  (17,042 rec/s)
  [PASS1]  829,000 / 1,987,897  (17,042 rec/s)
  [PASS1]  829,500 / 1,987,897  (17,043 rec/s)
  [PASS1]  830,000 / 1,987,897  (17,045 rec/s)
  [PASS1]  830,500 / 1,987,897  (17,046 rec/s)
  [PASS1]  831,000 / 1,987,897  (17,044 rec/s)
  [PASS1]  831,500 / 1,987,897  (17,045 rec/s)
  [PASS1]  832,000 / 1,987,897  (17,046 rec/s)
  [PASS1]  832,500 / 1,987,897  (17,046 rec/s)
  [PASS1]  833,000 / 1,987,897  (17,046 rec/s)
  [PASS1]  833,500 / 1,987,897  (17,048 rec/s)
  [PASS1]  834,000 / 1,987,897  (17,050 rec/s)
  [PASS1]  834,500 / 1,987,897  (17,050 rec/s)
  [PASS1]  835,000 / 1,987,897  (17,049 rec/s)
  [PASS1]  835,500 / 1,987,897  (17,050 rec/s)
  [PASS1]  836,000 / 1,987,897  (17,050 rec/s)
  [PASS1]  836,500 / 1,987,897  (17,050 rec/s)
  [PASS1]  83

In [4]:
# ============================================================
# PASS 2 — CDC Update: 30% users thay đổi
# Silver MERGE sẽ detect sự khác biệt và tạo version mới (SCD2 UPDATE)
# ============================================================
print("=" * 60)
pass2_updated = run_pass2(producer, all_users, pct=0.30, delay=0.002, seed=42)

print()
print("⏸  Chờ 30s để Silver pipeline xử lý MERGE...")
time.sleep(30)

PASS 2 — CDC UPDATE (596,369 users, 30%)
  [PASS2]      500 / 596,369  (427 rec/s)
  [PASS2]    1,000 / 596,369  (433 rec/s)
  [PASS2]    1,500 / 596,369  (432 rec/s)
  [PASS2]    2,000 / 596,369  (432 rec/s)
  [PASS2]    2,500 / 596,369  (434 rec/s)
  [PASS2]    3,000 / 596,369  (436 rec/s)
  [PASS2]    3,500 / 596,369  (438 rec/s)
  [PASS2]    4,000 / 596,369  (438 rec/s)
  [PASS2]    4,500 / 596,369  (439 rec/s)
  [PASS2]    5,000 / 596,369  (439 rec/s)
  [PASS2]    5,500 / 596,369  (440 rec/s)
  [PASS2]    6,000 / 596,369  (440 rec/s)
  [PASS2]    6,500 / 596,369  (440 rec/s)
  [PASS2]    7,000 / 596,369  (440 rec/s)
  [PASS2]    7,500 / 596,369  (440 rec/s)
  [PASS2]    8,000 / 596,369  (440 rec/s)
  [PASS2]    8,500 / 596,369  (419 rec/s)
  [PASS2]    9,000 / 596,369  (407 rec/s)
  [PASS2]    9,500 / 596,369  (400 rec/s)
  [PASS2]   10,000 / 596,369  (388 rec/s)
  [PASS2]   10,500 / 596,369  (376 rec/s)
  [PASS2]   11,000 / 596,369  (369 rec/s)
  [PASS2]   11,500 / 596,369  (369 

  [PASS2] ✅ Xong 596,369 records  |  1560.2s  |  382 rec/s  |  errors=0

⏸  Chờ 30s để Silver pipeline xử lý MERGE...


In [5]:
# ============================================================
# PASS 3 — CDC Update lần 2: 10% users (subset của Pass 2)
# Tạo version 3 trong SCD2 — chứng minh pipeline xử lý multi-version
# ============================================================
print("=" * 60)
pass3_updated = run_pass3(producer, all_users, pass2_updated, pct=0.10, delay=0.002)

print()
print("🎉 Hoàn tất 3 passes!")
print(f"   Pass 1 (full load) : {len(all_users):,} users")
print(f"   Pass 2 (update 30%): {len(pass2_updated):,} users")
print(f"   Pass 3 (update 10%): {len(pass3_updated):,} users")

PASS 3 — CDC UPDATE LẦN 2 (198,789 users)
  [PASS3]      500 / 198,789  (429 rec/s)
  [PASS3]    1,000 / 198,789  (434 rec/s)
  [PASS3]    1,500 / 198,789  (435 rec/s)
  [PASS3]    2,000 / 198,789  (435 rec/s)
  [PASS3]    2,500 / 198,789  (436 rec/s)
  [PASS3]    3,000 / 198,789  (397 rec/s)
  [PASS3]    3,500 / 198,789  (375 rec/s)
  [PASS3]    4,000 / 198,789  (359 rec/s)
  [PASS3]    4,500 / 198,789  (356 rec/s)
  [PASS3]    5,000 / 198,789  (350 rec/s)
  [PASS3]    5,500 / 198,789  (355 rec/s)
  [PASS3]    6,000 / 198,789  (361 rec/s)
  [PASS3]    6,500 / 198,789  (366 rec/s)
  [PASS3]    7,000 / 198,789  (371 rec/s)
  [PASS3]    7,500 / 198,789  (375 rec/s)
  [PASS3]    8,000 / 198,789  (379 rec/s)
  [PASS3]    8,500 / 198,789  (382 rec/s)
  [PASS3]    9,000 / 198,789  (385 rec/s)
  [PASS3]    9,500 / 198,789  (388 rec/s)
  [PASS3]   10,000 / 198,789  (390 rec/s)
  [PASS3]   10,500 / 198,789  (386 rec/s)
  [PASS3]   11,000 / 198,789  (379 rec/s)
  [PASS3]   11,500 / 198,789  (374

In [6]:
# ============================================================
# VERIFICATION — Xem sample data đã gửi
# ============================================================
import json

print("=== SAMPLE: Pass 1 (original) ===")
u_orig = all_users[0]
print(json.dumps(u_orig, indent=2))

print()
print("=== SAMPLE: Pass 2 (same user, updated) ===")
uid = pass2_updated[0]["user_id"]
orig = next((u for u in all_users if u["user_id"] == uid), None)
upd = pass2_updated[0]
if orig:
    print(f"user_id       : {uid}")
    print(f"review_count  : {orig['review_count']} → {upd['review_count']}  (+{upd['review_count']-orig['review_count']})")
    print(f"fans          : {orig['fans']} → {upd['fans']}  (+{upd['fans']-orig['fans']})")
    print(f"average_stars : {orig['average_stars']} → {upd['average_stars']}")
    print()
    print("✅ Silver MERGE sẽ detect sự thay đổi này và tạo SCD2 history record")

=== SAMPLE: Pass 1 (original) ===
{
  "average_stars": 3.91,
  "name": "Walker",
  "fans": 267,
  "review_count": 585,
  "yelping_since": "2007-01-25 16:47:26",
  "funny": 1259,
  "elite": "2007",
  "useful": 7217,
  "cool": 5994,
  "user_id": "qVc8ODYU5SZjKXVBgXdI7w"
}

=== SAMPLE: Pass 2 (same user, updated) ===
user_id       : bOahckJVnlWI4edsZF0pCQ
review_count  : 2 → 12  (+10)
fans          : 0 → 3  (+3)
average_stars : 3.0 → 3.02

✅ Silver MERGE sẽ detect sự thay đổi này và tạo SCD2 history record
